In [1]:
import xarray as xr
from dask.distributed import Client
from dask_jobqueue import PBSCluster
import matplotlib.pyplot as plt
import numpy as np

In [48]:
cluster = PBSCluster(
    cores=1,
    memory='40GB',
    processes=1,
    queue='casper',
    local_directory='$TMPDIR',
    account='P93300313',
    walltime='4:00:00'
)
cluster.scale(jobs=10)
client = Client(cluster)
client

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.185:38071,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [19]:
path = '/glade/work/acruz/Caribbean_Heat_data/ERA5/'
hi_ds = xr.open_dataset(path+'hourly_HI.nc')
u_ds  = xr.open_zarr(path+'U10')
v_ds  = xr.open_zarr(path+'V10')

In [20]:
u_ds = u_ds['VAR_10U'].sel(time=slice('1981', '2025'))
u_ds

<xarray.DataArray 'VAR_10U' (time: 394464, latitude: 82, longitude: 121)> Size: 16GB
dask.array<getitem, shape=(394464, 82, 121), dtype=float32, chunksize=(33052, 82, 121), chunktype=numpy.ndarray>
Coordinates:
  * time       (time) datetime64[ns] 3MB 1981-01-01 ... 2025-12-31T23:00:00
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0

In [21]:
v_ds = v_ds['VAR_10V'].sel(time=slice('1981', '2025'))
v_ds

<xarray.DataArray 'VAR_10V' (time: 394464, latitude: 82, longitude: 121)> Size: 16GB
dask.array<getitem, shape=(394464, 82, 121), dtype=float32, chunksize=(33052, 82, 121), chunktype=numpy.ndarray>
Coordinates:
  * time       (time) datetime64[ns] 3MB 1981-01-01 ... 2025-12-31T23:00:00
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0

In [22]:
# chunk for dask
hi_ds = hi_ds.chunk(time=17166)
hi_ds

<xarray.Dataset> Size: 30GB
Dimensions:    (time: 755304, latitude: 82, longitude: 121)
Coordinates:
  * time       (time) datetime64[ns] 6MB 1940-01-01 ... 2026-02-28T23:00:00
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0
Data variables:
    HI_hourly  (time, latitude, longitude) float32 30GB dask.array<chunksize=(17166, 82, 121), meta=np.ndarray>

In [23]:
# time range
hi_ds = hi_ds.sel(time=v_ds['time'])
hi_ds

<xarray.Dataset> Size: 16GB
Dimensions:    (time: 394464, latitude: 82, longitude: 121)
Coordinates:
  * time       (time) datetime64[ns] 3MB 1981-01-01 ... 2025-12-31T23:00:00
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0
Data variables:
    HI_hourly  (time, latitude, longitude) float32 16GB dask.array<chunksize=(17166, 82, 121), meta=np.ndarray>

In [28]:
# make hour dim lazily
hi_ds_coarse = hi_ds.coarsen(time=24, boundary='trim').construct(time=('day', 'hour'))
u_ds_coarse = u_ds.coarsen(time=24, boundary='trim').construct(time=('day', 'hour'))
v_ds_coarse = v_ds.coarsen(time=24, boundary='trim').construct(time=('day', 'hour'))

# get hour where max happens
hidmax_idx = hi_ds_coarse['HI_hourly'].argmax(dim='hour')

def sel_at_idx(ds, idx_ds):
    # add hour dimension to indexing array
    idx_ds = np.expand_dims(idx_ds, axis=-1)
    # select along hour axis and remove hour axis
    return np.take_along_axis(ds, idx_ds, axis=-1).squeeze(axis=-1)

In [43]:
u_peakHI = xr.apply_ufunc(sel_at_idx, u_ds_coarse, hidmax_idx,
                              input_core_dims=[['hour'], []],
                              output_core_dims=[[]],
                              dask='parallelized',
                              output_dtypes=[u_ds_coarse.dtype]
                             )
u_peakHI = u_peakHI.rename('U_during_HIdmax')
u_peakHI

<xarray.DataArray 'U_during_HIdmax' (day: 16436, latitude: 82, longitude: 121)> Size: 652MB
dask.array<transpose, shape=(16436, 82, 121), dtype=float32, chunksize=(715, 82, 121), chunktype=numpy.ndarray>
Coordinates:
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0
Dimensions without coordinates: day

In [44]:
v_peakHI = xr.apply_ufunc(sel_at_idx, v_ds_coarse, hidmax_idx,
                              input_core_dims=[['hour'], []],
                              output_core_dims=[[]],
                              dask='parallelized',
                              output_dtypes=[v_ds_coarse.dtype]
                             )
v_peakHI = v_peakHI.rename('V_during_HIdmax')
v_peakHI

<xarray.DataArray 'V_during_HIdmax' (day: 16436, latitude: 82, longitude: 121)> Size: 652MB
dask.array<transpose, shape=(16436, 82, 121), dtype=float32, chunksize=(715, 82, 121), chunktype=numpy.ndarray>
Coordinates:
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0
Dimensions without coordinates: day

In [45]:
timestamps = hi_ds['time'].isel(time=slice(0, None, 24)).dt.floor("1D")
timestamps

<xarray.DataArray 'floor' (time: 16436)> Size: 131kB
array(['1981-01-01T00:00:00.000000000', '1981-01-02T00:00:00.000000000',
       '1981-01-03T00:00:00.000000000', ...,
       '2025-12-29T00:00:00.000000000', '2025-12-30T00:00:00.000000000',
       '2025-12-31T00:00:00.000000000'],
      shape=(16436,), dtype='datetime64[ns]')
Coordinates:
  * time     (time) datetime64[ns] 131kB 1981-01-01 1981-01-02 ... 2025-12-31

In [46]:
u_peakHI = u_peakHI.rename({'day': 'time'}).assign_coords(day=timestamps).drop_vars('day')
v_peakHI = v_peakHI.rename({'day': 'time'}).assign_coords(day=timestamps).drop_vars('day')

In [49]:
u_peakHI.to_netcdf(path+'U10_during_peak_HI.nc', mode='w')

In [50]:
v_peakHI.to_netcdf(path+'V10_during_peak_HI.nc', mode='w')

In [51]:
client.shutdown()